# **CARGA Y TOKENIZACIÓN**

In [ ]:
#========================
# 0. Importar
#========================

import os
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Value, load_dataset
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    fbeta_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


In [ ]:
# ============================================================
# 0. ENTORNO / RUTAS
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

os.chdir('/content/drive/My Drive/TFG_COLAB/Etiquetas_TFG/')
print("Directorio de trabajo actual:", os.getcwd())
print("Archivos disponibles:", os.listdir())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de trabajo actual: /content/drive/My Drive/TFG_COLAB/Etiquetas_TFG
Archivos disponibles: ['Test_1305.csv', 'train_1305.csv', 'valid_1305.csv', 'outputs_phase1', 'outputs_phase2', 'corpus_1305.csv', 'corpus_sample.csv']


In [ ]:
# ============================================================
# 1. CARGA DE DATOS
# ============================================================

data_files = {
    "train": "train_1305.csv",
    "validation": "valid_1305.csv",
    "test": "Test_1305.csv",
    "corpus": "corpus_1305.csv",
    "corpus_sample": "corpus_sample.csv",
}

dict_TFG = load_dataset("csv", data_files=data_files, delimiter=';')
print(dict_TFG)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating corpus split: 0 examples [00:00, ? examples/s]

Generating corpus_sample split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos'],
        num_rows: 5226
    })
    validation: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos'],
        num_rows: 749
    })
    test: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos'],
        num_rows: 1474
    })
    corpus: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos'],
        num_rows: 1044147
    })
    corpus_sample: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos'],
        num_rows: 399
    })
})


In [ ]:
# ============================================================
# 2. TOKENIZACIÓN
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
MAX_LEN = 96

# IMPORTANTE:
# El plan metodológico habla de la variable `text`, pero tu código actual usa
# `text_en`. Dejamos ese comportamiento por coherencia con tu implementación.
# Si metodológicamente quieres seguir el plan de forma literal, cambia a `text`.
TEXT_COL = "text_en"

for split in dict_TFG.keys():
    if TEXT_COL not in dict_TFG[split].column_names:
        raise ValueError(
            f"El split '{split}' no contiene la columna '{TEXT_COL}'. "
            f"Columnas disponibles: {dict_TFG[split].column_names}"
        )


def tokenize_function(batch):
    texts = [str(text) for text in batch[TEXT_COL]]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )


tokenized_TFG = dict_TFG.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
)


print(tokenized_TFG)
print(tokenized_TFG["train"].column_names)


Map:   0%|          | 0/5226 [00:00<?, ? examples/s]

Map:   0%|          | 0/749 [00:00<?, ? examples/s]

Map:   0%|          | 0/1474 [00:00<?, ? examples/s]

Map:   0%|          | 0/1044147 [00:00<?, ? examples/s]

Map:   0%|          | 0/399 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5226
    })
    validation: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 749
    })
    test: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1474
    })
    corpus: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1044147
    })
    corpus_sample: Dataset({
        features: ['id', 'text_en', 'label', 'manifesto_id', 'pos', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 399
    })
})
['id', 'text_en', 'label', 'manifesto_id', 'pos', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# ============================================================
# 3. ASEGURAR ETIQUETAS
# ============================================================


def unify_label_column(dataset_dict):
    """Garantiza que la variable objetivo se llame `labels`."""
    for split in dataset_dict.keys():
        cols = dataset_dict[split].column_names

        if "labels" in cols:
            continue
        if "label" in cols:
            dataset_dict[split] = dataset_dict[split].rename_column("label", "labels")
            continue

        raise ValueError(
            f"El split '{split}' no contiene ni 'label' ni 'labels'. "
            f"Columnas disponibles: {cols}"
        )

    return dataset_dict


def ensure_binary_int_labels(dataset_dict):
    """
    Fuerza labels a int64 y comprueba que los valores sean estrictamente 0/1.
    """
    for split in dataset_dict.keys():
        dataset_dict[split] = dataset_dict[split].cast_column("labels", Value("int64"))
        unique_labels = set(dataset_dict[split]["labels"])

        if not unique_labels.issubset({0, 1}):
            raise ValueError(
                f"El split '{split}' contiene etiquetas no binarias: {sorted(unique_labels)}"
            )

        print(f"Split '{split}' -> etiquetas válidas: {sorted(unique_labels)}")

    return dataset_dict


tokenized_TFG = unify_label_column(tokenized_TFG)
tokenized_TFG = ensure_binary_int_labels(tokenized_TFG)


Casting the dataset:   0%|          | 0/5226 [00:00<?, ? examples/s]

Split 'train' -> etiquetas válidas: [0, 1]


Casting the dataset:   0%|          | 0/749 [00:00<?, ? examples/s]

Split 'validation' -> etiquetas válidas: [0, 1]


Casting the dataset:   0%|          | 0/1474 [00:00<?, ? examples/s]

Split 'test' -> etiquetas válidas: [0, 1]


Casting the dataset:   0%|          | 0/1044147 [00:00<?, ? examples/s]

Split 'corpus' -> etiquetas válidas: [0]


Casting the dataset:   0%|          | 0/399 [00:00<?, ? examples/s]

Split 'corpus_sample' -> etiquetas válidas: [0]


In [ ]:
# ============================================================
# 4. COLUMNAS PARA TRAINER
# ============================================================

model_columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_TFG["train"].column_names:
    model_columns.append("token_type_ids")

id_columns = [
    col for col in ["manifesto_id", "pos", "id", "text_en"]
    if col in tokenized_TFG["train"].column_names
]

# Dataset que entra al Trainer
trainer_TFG = tokenized_TFG.select_columns(model_columns)

# Comprobaciones
for split in trainer_TFG.keys():
    print(f"Columnas del modelo en {split}: {trainer_TFG[split].column_names}")

required_id_cols = ["manifesto_id", "pos"]
for split in tokenized_TFG.keys():
    missing = [col for col in required_id_cols if col not in tokenized_TFG[split].column_names]
    if missing:
        raise ValueError(f"Faltan columnas de identificación en {split}: {missing}")

print(f"Columnas auxiliares disponibles para agregación posterior: {id_columns}")


Columnas del modelo en train: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
Columnas del modelo en validation: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
Columnas del modelo en test: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
Columnas del modelo en corpus: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
Columnas del modelo en corpus_sample: ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
Columnas auxiliares disponibles para agregación posterior: ['manifesto_id', 'pos', 'id', 'text_en']


# **AJUSTE DEL MODELO**

In [ ]:
# ============================================================
# 5. SEMILLA Y MODELO
# ============================================================

SEED = 28042026


def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_all_seeds(SEED)

label2id = {"negative": 0, "positive": 1}
id2label = {0: "negative", 1: "positive"}


def model_init():
    set_all_seeds(SEED)
    return AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )


In [ ]:
# ============================================================
# 6. MÉTRICAS Y UMBRALES
# ============================================================

# Coherente con la malla del plan metodológico
THRESHOLDS = np.array([0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80])


def softmax_np(logits):
    logits = np.asarray(logits)
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    return probs


def extract_logits(pred_output):
    return pred_output.predictions[0] if isinstance(pred_output.predictions, tuple) else pred_output.predictions


def metrics_at_threshold(y_true, prob_pos, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos)

    y_pred = (prob_pos >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)

    precision_pos, recall_pos, f1_pos, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    # Lo mantengo como métrica diagnóstica, aunque ya no será criterio principal
    f01_pos = fbeta_score(
        y_true,
        y_pred,
        beta=0.1,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    prevalence_real = np.mean(y_true)
    prevalence_pred = np.mean(y_pred)
    abs_prevalence_error = abs(prevalence_pred - prevalence_real)

    return {
        "threshold": float(threshold),
        "accuracy": float(acc),
        "precision_pos": float(precision_pos),
        "recall_pos": float(recall_pos),
        "f1_pos": float(f1_pos),
        "f01_pos": float(f01_pos),
        "fpr": float(fpr),
        "prevalence_real": float(prevalence_real),
        "prevalence_pred": float(prevalence_pred),
        "abs_prevalence_error": float(abs_prevalence_error),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def select_best_threshold(y_true, prob_pos, thresholds=THRESHOLDS):
    """
    Selecciona el mejor umbral con esta jerarquía:
    1. Mayor precisión positiva
    2. Mayor F1 positivo
    3. Mayor umbral
    """
    results = [metrics_at_threshold(y_true, prob_pos, thr) for thr in thresholds]

    results_sorted = sorted(
        results,
        key=lambda x: (
            x["precision_pos"],
            x["f1_pos"],
            x["threshold"]
        ),
        reverse=True
    )

    best_result = results_sorted[0]
    return best_result, results


def compute_metrics(pred):
    """
    Métricas para Trainer.
    El checkpoint 'mejor' quedará definido por precision_pos_best.
    """
    y_true = pred.label_ids
    logits = pred.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    probs = softmax_np(logits)
    prob_pos = probs[:, 1]

    best_metrics, _ = select_best_threshold(
        y_true=y_true,
        prob_pos=prob_pos,
        thresholds=THRESHOLDS
    )

    return {
        "best_threshold": best_metrics["threshold"],
        "precision_pos_best": best_metrics["precision_pos"],
        "recall_pos_best": best_metrics["recall_pos"],
        "f1_pos_best": best_metrics["f1_pos"],
        "f01_pos_best": best_metrics["f01_pos"],
        "fpr_best": best_metrics["fpr"],
        "prevalence_pred_best": best_metrics["prevalence_pred"],
        "abs_prevalence_error_best": best_metrics["abs_prevalence_error"]
    }


def get_validation_threshold_table(pred_output):
    logits = extract_logits(pred_output)
    y_true = pred_output.label_ids
    prob_pos = softmax_np(logits)[:, 1]

    best_result, all_results = select_best_threshold(y_true, prob_pos, thresholds=THRESHOLDS)

    threshold_df = pd.DataFrame(all_results).sort_values(
        by=["precision_pos", "f1_pos", "threshold"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    return best_result, threshold_df


In [ ]:
# ============================================================
# 7. FASE 1: REJILLA DE HIPERPARÁMETROS
# ============================================================

# Nueva estrategia:
# - Fase 1 explora learning rate x weight_decay
# - Se mantiene fijo num_train_epochs para esta fase
PHASE1_FIXED_NUM_EPOCHS = 3
PHASE1_WEIGHT_DECAYS = [0.0, 0.01, 0.05]
PHASE1_LEARNING_RATES = [1e-5, 2e-5, 3e-5, 5e-5]

PHASE1_GRID = []
model_counter = 1
for wd in PHASE1_WEIGHT_DECAYS:
    for lr in PHASE1_LEARNING_RATES:
        PHASE1_GRID.append({
            "model_id": f"M{model_counter}",
            "learning_rate": lr,
            "weight_decay": wd,
            "num_train_epochs": PHASE1_FIXED_NUM_EPOCHS,
        })
        model_counter += 1

# Carpeta de salida
OUTPUT_ROOT = Path("./outputs_phase1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Rejilla Fase 1:")
print(pd.DataFrame(PHASE1_GRID).to_string(index=False))


In [ ]:
# ============================================================
# 8. HIPERPARÁMETROS Y FUNCIÓN DE ENTRENAMIENTO
# ============================================================


def build_training_arguments(run_cfg):
    output_dir = OUTPUT_ROOT / run_cfg["model_id"]

    return TrainingArguments(
        output_dir=str(output_dir),
        do_train=True,
        do_eval=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=run_cfg["num_train_epochs"],
        learning_rate=run_cfg["learning_rate"],
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        weight_decay=run_cfg["weight_decay"],
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="precision_pos_best",
        greater_is_better=True,
        save_total_limit=1,
        seed=SEED,
        data_seed=SEED,
        report_to="none"
    )


def resolve_checkpoint_path(trainer, model_dir):
    """
    Garantiza que exista una ruta cargable para el modelo entrenado.

    Prioridad:
    1. best_model_checkpoint del Trainer
    2. último checkpoint encontrado en disco
    3. guardado explícito del modelo final como fallback
    """
    best_checkpoint = trainer.state.best_model_checkpoint
    best_metric = trainer.state.best_metric

    if best_checkpoint is not None and Path(best_checkpoint).exists():
        print(f"Checkpoint óptimo guardado correctamente: {best_checkpoint}")
        return str(best_checkpoint), "best_model_checkpoint", best_metric, None

    checkpoint_warning = None

    if best_checkpoint is None:
        checkpoint_warning = (
            "Trainer no registró `best_model_checkpoint`. "
            "Se intentará recuperar el último checkpoint disponible."
        )
    else:
        checkpoint_warning = (
            f"`best_model_checkpoint` apunta a una ruta inexistente: {best_checkpoint}. "
            "Se intentará recuperar el último checkpoint disponible."
        )

    checkpoint_dirs = sorted(
        [p for p in model_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: p.name
    )

    if checkpoint_dirs:
        fallback_checkpoint = str(checkpoint_dirs[-1])
        print(checkpoint_warning)
        print(f"Se usará el último checkpoint encontrado: {fallback_checkpoint}")
        return fallback_checkpoint, "last_checkpoint_on_disk", best_metric, checkpoint_warning

    fallback_dir = model_dir / "final_model_fallback"
    fallback_dir.mkdir(parents=True, exist_ok=True)

    trainer.save_model(str(fallback_dir))
    tokenizer.save_pretrained(str(fallback_dir))

    checkpoint_warning = (
        (checkpoint_warning or "")
        + " No se encontró ningún directorio `checkpoint-*`, así que se guardó el "
          "modelo final manualmente en `final_model_fallback`."
    )

    print(checkpoint_warning)
    print(f"Se usará el fallback manual: {fallback_dir}")

    return str(fallback_dir), "manual_fallback_save", best_metric, checkpoint_warning


def run_phase1_model(run_cfg, return_trainer=False):
    set_all_seeds(SEED)

    args = build_training_arguments(run_cfg)
    model_dir = OUTPUT_ROOT / run_cfg["model_id"]
    model_dir.mkdir(parents=True, exist_ok=True)

    trainer = Trainer(
        model_init=model_init,
        args=args,
        train_dataset=trainer_TFG["train"],
        eval_dataset=trainer_TFG["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(
        f"\n=== Ejecutando {run_cfg['model_id']} | "
        f"lr={run_cfg['learning_rate']} | "
        f"weight_decay={run_cfg['weight_decay']} | "
        f"epochs={run_cfg['num_train_epochs']} ==="
    )

    train_output = trainer.train()
    eval_metrics = trainer.evaluate()
    val_pred = trainer.predict(trainer_TFG["validation"])

    best_threshold_result, threshold_df = get_validation_threshold_table(val_pred)

    resolved_checkpoint, checkpoint_source, resolved_metric, checkpoint_warning = resolve_checkpoint_path(
        trainer=trainer,
        model_dir=model_dir,
    )

    threshold_path = model_dir / "validation_threshold_scan.csv"
    metrics_path = model_dir / "validation_summary.json"

    threshold_df.to_csv(threshold_path, index=False)

    summary = {
        "run_config": run_cfg,
        "selection_rule": {
            "primary": "precision_pos",
            "secondary": "f1_pos",
            "tertiary": "threshold"
        },
        "train_runtime_seconds": float(train_output.metrics.get("train_runtime", 0.0)),
        "best_checkpoint": resolved_checkpoint,
        "best_checkpoint_source": checkpoint_source,
        "best_metric_from_trainer": float(resolved_metric) if resolved_metric is not None else None,
        "checkpoint_warning": checkpoint_warning,
        "eval_metrics": eval_metrics,
        "best_threshold_result": best_threshold_result,
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nResumen validación:")
    print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

    print("\nTop umbrales en validación:")
    print(
        threshold_df[
            [
                "threshold",
                "precision_pos",
                "f1_pos",
                "recall_pos",
                "fpr",
                "abs_prevalence_error",
            ]
        ].to_string(index=False)
    )

    result = {
        "summary": summary,
        "threshold_table": threshold_df,
    }

    if return_trainer:
        result["trainer"] = trainer

    return result


In [ ]:
# ============================================================
# 9. FASE 1 COMPLETA: EJECUCIÓN DE LA REJILLA Y RANKING
# ============================================================

PHASE1_MASTER_CSV = OUTPUT_ROOT / "phase1_results.csv"
PHASE1_MASTER_JSON = OUTPUT_ROOT / "phase1_results.json"
PHASE1_TOP2_JSON = OUTPUT_ROOT / "phase1_top2.json"


def phase1_summary_to_row(summary):
    run_cfg = summary["run_config"]
    best = summary["best_threshold_result"]
    eval_metrics = summary["eval_metrics"]

    return {
        "model_id": run_cfg["model_id"],
        "learning_rate": run_cfg["learning_rate"],
        "weight_decay": run_cfg["weight_decay"],
        "num_train_epochs": run_cfg["num_train_epochs"],
        "best_threshold": best["threshold"],
        "precision_pos_best": best["precision_pos"],
        "f1_pos_best": best["f1_pos"],
        "recall_pos_best": best["recall_pos"],
        "f01_pos_best": best["f01_pos"],
        "fpr_best": best["fpr"],
        "prevalence_pred_best": best["prevalence_pred"],
        "abs_prevalence_error_best": best["abs_prevalence_error"],
        "trainer_best_metric": summary["best_metric_from_trainer"],
        "best_checkpoint": summary["best_checkpoint"],
        "best_checkpoint_source": summary["best_checkpoint_source"],
        "checkpoint_warning": summary["checkpoint_warning"],
        "eval_loss": eval_metrics.get("eval_loss"),
        "train_runtime_seconds": summary["train_runtime_seconds"],
    }


def rank_phase1_results(df):
    return df.sort_values(
        by=["precision_pos_best", "f1_pos_best", "best_threshold"],
        ascending=[False, False, False]
    ).reset_index(drop=True)


def run_full_phase1_grid(grid=PHASE1_GRID):
    all_summaries = []
    all_rows = []

    for run_cfg in grid:
        result = run_phase1_model(run_cfg, return_trainer=False)
        summary = result["summary"]

        all_summaries.append(summary)
        all_rows.append(phase1_summary_to_row(summary))

        partial_df = rank_phase1_results(pd.DataFrame(all_rows))
        partial_df.to_csv(PHASE1_MASTER_CSV, index=False)

        with open(PHASE1_MASTER_JSON, "w", encoding="utf-8") as f:
            json.dump(all_summaries, f, ensure_ascii=False, indent=2, default=str)

        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    phase1_df = rank_phase1_results(pd.DataFrame(all_rows))
    top2_df = phase1_df.head(2).copy()

    phase1_df.to_csv(PHASE1_MASTER_CSV, index=False)

    with open(PHASE1_MASTER_JSON, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, ensure_ascii=False, indent=2, default=str)

    with open(PHASE1_TOP2_JSON, "w", encoding="utf-8") as f:
        json.dump(top2_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2, default=str)

    print("\n=== Ranking final Fase 1 ===")
    print(
        phase1_df[
            [
                "model_id",
                "learning_rate",
                "weight_decay",
                "num_train_epochs",
                "best_threshold",
                "precision_pos_best",
                "f1_pos_best",
                "recall_pos_best",
                "fpr_best",
                "abs_prevalence_error_best",
                "best_checkpoint_source",
            ]
        ].to_string(index=False)
    )

    print("\nTop 2 para pasar a Fase 2:")
    print(
        top2_df[
            [
                "model_id",
                "learning_rate",
                "weight_decay",
                "best_threshold",
                "precision_pos_best",
                "f1_pos_best",
                "best_checkpoint_source",
            ]
        ].to_string(index=False)
    )

    return phase1_df, top2_df


In [ ]:
# ============================================================
# 10. EJECUCIÓN DE LA FASE 1 COMPLETA
# ============================================================

phase1_results_df, phase1_top2_df = run_full_phase1_grid(PHASE1_GRID)
print("\nResultados completos de Fase 1:")
print(phase1_results_df.to_string(index=False))


In [ ]:
# ============================================================
# 11. FASE 2: REJILLA DE HIPERPARÁMETROS Y EJECUCIÓN
# ============================================================

# Nueva estrategia:
# - Se toman los dos mejores modelos de Fase 1
# - Se mantienen learning_rate y weight_decay del modelo base
# - Se refinan batch size, warmup ratio y num_train_epochs
PHASE2_BATCH_SIZES = [16, 32]
PHASE2_WARMUP_RATIOS = [0.10, 0.06]
PHASE2_NUM_EPOCHS = [2, 3]

PHASE2_GRID = []
for _, row in phase1_top2_df.iterrows():
    base_model_id = row["model_id"]
    lr = float(row["learning_rate"])
    wd = float(row["weight_decay"])

    run_counter = 1
    for batch_size in PHASE2_BATCH_SIZES:
        for warmup_ratio in PHASE2_WARMUP_RATIOS:
            for num_epochs in PHASE2_NUM_EPOCHS:
                PHASE2_GRID.append({
                    "model_id": f"{base_model_id}_R{run_counter}",
                    "base_model_id": base_model_id,
                    "learning_rate": lr,
                    "weight_decay": wd,
                    "per_device_train_batch_size": batch_size,
                    "warmup_ratio": warmup_ratio,
                    "num_train_epochs": num_epochs,
                })
                run_counter += 1

OUTPUT_ROOT_PHASE2 = Path("./outputs_phase2")
OUTPUT_ROOT_PHASE2.mkdir(parents=True, exist_ok=True)

print("Rejilla Fase 2:")
print(pd.DataFrame(PHASE2_GRID).to_string(index=False))


def build_training_arguments_phase2(run_cfg):
    output_dir = OUTPUT_ROOT_PHASE2 / run_cfg["model_id"]

    return TrainingArguments(
        output_dir=str(output_dir),
        do_train=True,
        do_eval=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=run_cfg["num_train_epochs"],
        learning_rate=run_cfg["learning_rate"],
        per_device_train_batch_size=run_cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=32,
        warmup_ratio=run_cfg["warmup_ratio"],
        weight_decay=run_cfg["weight_decay"],
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="precision_pos_best",
        greater_is_better=True,
        save_total_limit=1,
        seed=SEED,
        data_seed=SEED,
        report_to="none"
    )


def run_phase2_model(run_cfg, return_trainer=False):
    set_all_seeds(SEED)

    args = build_training_arguments_phase2(run_cfg)
    model_dir = OUTPUT_ROOT_PHASE2 / run_cfg["model_id"]
    model_dir.mkdir(parents=True, exist_ok=True)

    trainer = Trainer(
        model_init=model_init,
        args=args,
        train_dataset=trainer_TFG["train"],
        eval_dataset=trainer_TFG["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(
        f"\n=== Ejecutando {run_cfg['model_id']} | base={run_cfg['base_model_id']} | "
        f"lr={run_cfg['learning_rate']} | weight_decay={run_cfg['weight_decay']} | "
        f"bs={run_cfg['per_device_train_batch_size']} | warmup_ratio={run_cfg['warmup_ratio']} | "
        f"epochs={run_cfg['num_train_epochs']} ==="
    )

    train_output = trainer.train()
    eval_metrics = trainer.evaluate()
    val_pred = trainer.predict(trainer_TFG["validation"])

    best_threshold_result, threshold_df = get_validation_threshold_table(val_pred)

    resolved_checkpoint, checkpoint_source, resolved_metric, checkpoint_warning = resolve_checkpoint_path(
        trainer=trainer,
        model_dir=model_dir,
    )

    threshold_path = model_dir / "validation_threshold_scan.csv"
    metrics_path = model_dir / "validation_summary.json"

    threshold_df.to_csv(threshold_path, index=False)

    summary = {
        "run_config": run_cfg,
        "selection_rule": {
            "primary": "precision_pos",
            "secondary": "f1_pos",
            "tertiary": "threshold"
        },
        "train_runtime_seconds": float(train_output.metrics.get("train_runtime", 0.0)),
        "best_checkpoint": resolved_checkpoint,
        "best_checkpoint_source": checkpoint_source,
        "best_metric_from_trainer": float(resolved_metric) if resolved_metric is not None else None,
        "checkpoint_warning": checkpoint_warning,
        "eval_metrics": eval_metrics,
        "best_threshold_result": best_threshold_result,
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nResumen validación:")
    print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

    print("\nTop umbrales en validación:")
    print(
        threshold_df[
            [
                "threshold",
                "precision_pos",
                "f1_pos",
                "recall_pos",
                "fpr",
                "abs_prevalence_error",
            ]
        ].to_string(index=False)
    )

    result = {
        "summary": summary,
        "threshold_table": threshold_df,
    }

    if return_trainer:
        result["trainer"] = trainer

    return result


def phase2_summary_to_row(summary):
    run_cfg = summary["run_config"]
    best = summary["best_threshold_result"]
    eval_metrics = summary["eval_metrics"]

    return {
        "model_id": run_cfg["model_id"],
        "base_model_id": run_cfg["base_model_id"],
        "learning_rate": run_cfg["learning_rate"],
        "weight_decay": run_cfg["weight_decay"],
        "per_device_train_batch_size": run_cfg["per_device_train_batch_size"],
        "warmup_ratio": run_cfg["warmup_ratio"],
        "num_train_epochs": run_cfg["num_train_epochs"],
        "best_threshold": best["threshold"],
        "precision_pos_best": best["precision_pos"],
        "f1_pos_best": best["f1_pos"],
        "recall_pos_best": best["recall_pos"],
        "f01_pos_best": best["f01_pos"],
        "fpr_best": best["fpr"],
        "prevalence_pred_best": best["prevalence_pred"],
        "abs_prevalence_error_best": best["abs_prevalence_error"],
        "trainer_best_metric": summary["best_metric_from_trainer"],
        "best_checkpoint": summary["best_checkpoint"],
        "best_checkpoint_source": summary["best_checkpoint_source"],
        "checkpoint_warning": summary["checkpoint_warning"],
        "eval_loss": eval_metrics.get("eval_loss"),
        "train_runtime_seconds": summary["train_runtime_seconds"],
    }


def rank_phase2_results(df):
    return df.sort_values(
        by=["precision_pos_best", "f1_pos_best", "best_threshold"],
        ascending=[False, False, False]
    ).reset_index(drop=True)


def run_full_phase2_grid(grid=PHASE2_GRID):
    all_summaries = []
    all_rows = []

    PHASE2_MASTER_CSV = OUTPUT_ROOT_PHASE2 / "phase2_results.csv"
    PHASE2_MASTER_JSON = OUTPUT_ROOT_PHASE2 / "phase2_results.json"

    for run_cfg in grid:
        result = run_phase2_model(run_cfg, return_trainer=False)
        summary = result["summary"]

        all_summaries.append(summary)
        all_rows.append(phase2_summary_to_row(summary))

        partial_df = rank_phase2_results(pd.DataFrame(all_rows))
        partial_df.to_csv(PHASE2_MASTER_CSV, index=False)

        with open(PHASE2_MASTER_JSON, "w", encoding="utf-8") as f:
            json.dump(all_summaries, f, ensure_ascii=False, indent=2, default=str)

        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    phase2_df = rank_phase2_results(pd.DataFrame(all_rows))
    phase2_df.to_csv(PHASE2_MASTER_CSV, index=False)

    with open(PHASE2_MASTER_JSON, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, ensure_ascii=False, indent=2, default=str)

    print("\n=== Ranking final Fase 2 ===")
    print(
        phase2_df[
            [
                "model_id",
                "base_model_id",
                "learning_rate",
                "weight_decay",
                "per_device_train_batch_size",
                "warmup_ratio",
                "num_train_epochs",
                "best_threshold",
                "precision_pos_best",
                "f1_pos_best",
                "recall_pos_best",
                "fpr_best",
                "abs_prevalence_error_best",
                "best_checkpoint_source",
            ]
        ].to_string(index=False)
    )

    return phase2_df

phase2_results_df = run_full_phase2_grid(PHASE2_GRID)


# **EVALUACIÓN**

In [ ]:
# ============================================================
# 12. CARGAR MODELO GANADOR FINAL DESDE phase2_results.json
# ============================================================

OUTPUT_ROOT_PHASE2 = Path("./outputs_phase2")
OUTPUT_ROOT_PHASE2.mkdir(parents=True, exist_ok=True)

# Ruta al archivo de resultados de Fase 2
PHASE2_RESULTS_JSON = OUTPUT_ROOT_PHASE2/"phase2_results.json"

if not PHASE2_RESULTS_JSON.exists():
    raise FileNotFoundError(
        f"No se encuentra el archivo de resultados de Fase 2:\n{PHASE2_RESULTS_JSON}\n\n"
        "Debes haber ejecutado Fase 2 al menos una vez y haber guardado phase2_results.json."
    )

# ------------------------------------------------------------
# 12.1. Cargar resultados de Fase 2
# ------------------------------------------------------------

with open(PHASE2_RESULTS_JSON, "r", encoding="utf-8") as f:
    phase2_results = json.load(f)

if not isinstance(phase2_results, list) or len(phase2_results) == 0:
    raise ValueError(
        "phase2_results.json no tiene el formato esperado: "
        "debe ser una lista no vacía de resultados."
    )

# ------------------------------------------------------------
# 12.2. Seleccionar modelo ganador según la regla de selección
# ------------------------------------------------------------

def phase2_ranking_key(result):
    best = result["best_threshold_result"]

    return (
        best["precision_pos"],
        best["f1_pos"],
        best["threshold"],
    )

winner_summary = max(phase2_results, key=phase2_ranking_key)

run_cfg = winner_summary["run_config"]
best_threshold_result = winner_summary["best_threshold_result"]

In [ ]:
# ------------------------------------------------------------
# 12.3. Recuperar configuración, checkpoint y umbral
# ------------------------------------------------------------

FINAL_RUN = {
    "model_id": run_cfg["model_id"],
    "base_model_id": run_cfg["base_model_id"],
    "learning_rate": float(run_cfg["learning_rate"]),
    "weight_decay": float(run_cfg["weight_decay"]),
    "per_device_train_batch_size": int(run_cfg["per_device_train_batch_size"]),
    "warmup_ratio": float(run_cfg["warmup_ratio"]),
    "num_train_epochs": int(run_cfg["num_train_epochs"]),
}

threshold = float(best_threshold_result["threshold"])

final_checkpoint = winner_summary["best_checkpoint"]

if final_checkpoint is None or str(final_checkpoint).strip() == "":
    raise RuntimeError(
        "No hay una ruta válida en best_checkpoint para el modelo ganador."
    )

final_checkpoint = Path(final_checkpoint)

if not final_checkpoint.exists():
    raise FileNotFoundError(
        f"La ruta del checkpoint final no existe:\n{final_checkpoint}\n\n"
        "El JSON conserva la ruta, pero los pesos del modelo deben seguir "
        "existiendo en esa carpeta."
    )

# ------------------------------------------------------------
# 12.4. Cargar modelo ganador sin reentrenar
# ------------------------------------------------------------

final_model = AutoModelForSequenceClassification.from_pretrained(
    final_checkpoint,
    local_files_only=True
)

final_model.eval()

print("Modelo ganador cargado correctamente sin reentrenar.")
print("\nConfiguración ganadora:")
print(json.dumps(FINAL_RUN, ensure_ascii=False, indent=2))

print("\nCheckpoint cargado:")
print(final_checkpoint)

print("\nUmbral óptimo:")
print(threshold)

print("\nMétricas de validación del ganador:")
print(json.dumps(best_threshold_result, ensure_ascii=False, indent=2))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo ganador cargado correctamente sin reentrenar.

Configuración ganadora:
{
  "model_id": "M1_R1",
  "base_model_id": "M1",
  "learning_rate": 1e-05,
  "weight_decay": 0.0,
  "per_device_train_batch_size": 16,
  "warmup_ratio": 0.1,
  "num_train_epochs": 2
}

Checkpoint cargado:
outputs_phase2/M1_R1/checkpoint-327

Umbral óptimo:
0.8

Métricas de validación del ganador:
{
  "threshold": 0.8,
  "accuracy": 0.9959946595460614,
  "precision_pos": 0.9919786096256684,
  "recall_pos": 1.0,
  "f1_pos": 0.9959731543624161,
  "f01_pos": 0.9920573985332662,
  "fpr": 0.007936507936507936,
  "prevalence_real": 0.4953271028037383,
  "prevalence_pred": 0.4993324432576769,
  "abs_prevalence_error": 0.004005340453938577,
  "tn": 375,
  "fp": 3,
  "fn": 0,
  "tp": 371
}


In [ ]:
#==========================================
#13. MODELO PARA EVALUAR
#==========================================
final_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT_PHASE2 / "final_analysis"),
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=32,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer_eval = Trainer(
    model=final_model,
    args=final_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# =========================================
# 14. DISTRIBUCIÓN DE PROBABILIDADES POSITIVAS
# =========================================

val_pred = trainer_eval.predict(trainer_TFG["validation"])

logits = extract_logits(val_pred)
prob_pos = softmax_np(logits)[:, 1]
y_true = np.array(val_pred.label_ids)

prob_series = pd.Series(prob_pos, name="prob_pos")

print("\nNúmero de casos en zonas clave (validación):")
print("0.05 <= prob_pos < 0.95:", int(((prob_pos >= 0.05) & (prob_pos < 0.95)).sum()))
print("0.001 <= prob_pos < 0.05:", int(((prob_pos >= 0.001) & (prob_pos < 0.05)).sum()))
print("prob_pos >= 0.95:", int((prob_pos >= 0.95).sum()))

# =========================================
# 15. RECUPERAR ERRORES DE CLASIFICACIÓN EN VALIDACIÓN
# =========================================

threshold = winner_run_result["summary"]["best_threshold_result"]["threshold"]
y_pred = (prob_pos >= threshold).astype(int)

val_meta = tokenized_TFG["validation"]

errors = []
for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        errors.append({
            "idx": val_meta[i]["id"] if "id" in val_meta.column_names else None,
            "manifesto_id": val_meta[i]["manifesto_id"] if "manifesto_id" in val_meta.column_names else None,
            "pos": val_meta[i]["pos"] if "pos" in val_meta.column_names else None,
            "y_true": int(y_true[i]),
            "y_pred": int(y_pred[i]),
            "prob_pos": float(prob_pos[i]),
            "error_type": "FP" if (y_true[i] == 0 and y_pred[i] == 1) else "FN",
            "text": val_meta[i]["text"] if "text" in val_meta.column_names else None,
            "text_en": val_meta[i]["text_en"] if "text_en" in val_meta.column_names else None,
        })

errors_df = pd.DataFrame(errors)

print("\nNúmero de errores en validación:", len(errors_df))
if errors_df.empty:
    print("No se detectaron errores de clasificación en validación.")
else:
    print(
        errors_df[
            ["idx", "text_en", "manifesto_id", "pos", "y_true", "y_pred", "prob_pos", "error_type"]
        ].to_string(index=False)
    )



Número de casos en zonas clave (validación):
0.05 <= prob_pos < 0.95: 1
0.001 <= prob_pos < 0.05: 375
prob_pos >= 0.95: 373

Número de errores en validación: 3
   idx                                                                                                               text_en manifesto_id  pos  y_true  y_pred  prob_pos error_type
236276                                               Higher activity rates are essential to keeping our pensions affordable. 21421_201006 1054       0       1  0.993398         FP
107219                                                                                         ABOLITION OF EARLY RETIREMENT 13001_201109   18       0       1  0.994523         FP
144826 This is particularly worrying for people with high levels of illness and high levels of need, including older people. 14820_201904  731       0       1  0.995659         FP


In [ ]:
# =========================================
# 16. EVALUACIÓN FINAL EN TEST
# =========================================

threshold = winner_run_result["summary"]["best_threshold_result"]["threshold"]

test_pred = trainer_eval.predict(trainer_TFG["test"])

logits_test = extract_logits(test_pred)
y_true_test = np.array(test_pred.label_ids)
prob_pos_test = softmax_np(logits_test)[:, 1]

test_metrics_test = metrics_at_threshold(
    y_true=y_true_test,
    prob_pos=prob_pos_test,
    threshold=threshold
)

print("\nUmbral fijado desde validación:", threshold)
print(json.dumps(test_metrics_test, ensure_ascii=False, indent=2))

# =========================================
# 17. RECUPERAR ERRORES DE CLASIFICACIÓN EN TEST
# =========================================

y_pred_test = (prob_pos_test >= threshold).astype(int)
test_meta = tokenized_TFG["test"]

errors_test = []
for i in range(len(y_true_test)):
    if y_true_test[i] != y_pred_test[i]:
        errors_test.append({
            "idx": test_meta[i]["id"] if "id" in test_meta.column_names else None,
            "manifesto_id": test_meta[i]["manifesto_id"] if "manifesto_id" in test_meta.column_names else None,
            "pos": test_meta[i]["pos"] if "pos" in test_meta.column_names else None,
            "y_true": int(y_true_test[i]),
            "y_pred_test": int(y_pred_test[i]),
            "prob_pos": float(prob_pos_test[i]),
            "error_type": "FP" if (y_true_test[i] == 0 and y_pred_test[i] == 1) else "FN",
            "text": test_meta[i]["text"] if "text" in test_meta.column_names else None,
            "text_en": test_meta[i]["text_en"] if "text_en" in test_meta.column_names else None,
        })

errors_df_test = pd.DataFrame(errors_test)

print("\nNúmero de errores en test:", len(errors_df_test))
if errors_df_test.empty:
    print("No se detectaron errores de clasificación en test.")
else:
    print(
        errors_df_test[
            ["idx", "text_en", "manifesto_id", "pos", "y_true", "y_pred_test", "prob_pos", "error_type"]
        ].to_string(index=False)
    )

prob_series_test = pd.Series(prob_pos_test, name="prob_pos")

print("\nNúmero de casos en zonas clave (test):")
print("0.05 <= prob_pos < 0.95:", int(((prob_pos_test >= 0.05) & (prob_pos_test < 0.95)).sum()))
print("0.001 <= prob_pos < 0.05:", int(((prob_pos_test >= 0.001) & (prob_pos_test < 0.05)).sum()))
print("prob_pos >= 0.95:", int((prob_pos_test >= 0.95).sum()))



Umbral fijado desde validación: 0.8
{
  "threshold": 0.8,
  "accuracy": 0.9925373134328358,
  "precision_pos": 0.864406779661017,
  "recall_pos": 0.9444444444444444,
  "f1_pos": 0.9026548672566371,
  "f01_pos": 0.8651326839099764,
  "fpr": 0.005633802816901409,
  "prevalence_real": 0.036635006784260515,
  "prevalence_pred": 0.04002713704206241,
  "abs_prevalence_error": 0.003392130257801898,
  "tn": 1412,
  "fp": 8,
  "fn": 3,
  "tp": 51
}

Número de errores en test: 11
    idx                                                                                                                                                                                                                                                                                               text_en manifesto_id  pos  y_true  y_pred_test  prob_pos error_type
 157471 Ecolo also believes that, in general, diversity in the workplace needs to be enhanced, and that discrimination against women, older workers, disabled peop

# **INFERENCIA**

In [ ]:
# ============================================================
# 19. APLICAR MODELO GANADOR AL CORPUS COMPLETO
# ============================================================

inference_args = TrainingArguments(
    output_dir=str(Path("outputs_phase2") / "corpus_inference"),
    do_train=False,
    do_eval=False,
    per_device_eval_batch_size=64,
    report_to="none",
)

inference_trainer = Trainer(
    model=final_model,
    args=inference_args,
    data_collator=data_collator,
)

#repito función para no cargar sección de evaluación
def softmax_np(logits):
    logits = np.asarray(logits)
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    return probs


corpus_sample_pred = inference_trainer.predict(trainer_TFG["corpus_sample"])

logits_corpus_sample = extract_logits(corpus_sample_pred)
y_true_corpus_sample = np.array(corpus_sample_pred.label_ids)
prob_pos_corpus_sample = softmax_np(logits_corpus_sample)[:, 1]

pred_label = (prob_pos_corpus_sample >= threshold).astype(int)

print("\nPredicción completada.")
print("Umbral aplicado:", threshold)
print("Proporción predicha como tópico:", pred_label.mean())


Predicción completada.
Umbral aplicado: 0.8
Proporción predicha como tópico: 0.020050125313283207


In [ ]:
# ============================================================
# 20. AÑADIR PREDICCIONES AL CORPUS REDUCIDO
# ============================================================

corpus_sample_with_meta = tokenized_TFG["corpus_sample"]

cols_to_remove = [
    col for col in [
        "input_ids",
        "attention_mask",
        "token_type_ids"
    ]
    if col in corpus_sample_with_meta.column_names
]

corpus_sample_df = (
    corpus_sample_with_meta
    .remove_columns(cols_to_remove)
    .to_pandas()
)

# Crear identificador interno por seguridad
if "row_id_corpus_sample" not in corpus_sample_df.columns:
    corpus_sample_df.insert(
        0,
        "row_id_corpus_sample",
        np.arange(len(corpus_sample_df))
    )

if len(corpus_sample_df) != len(pred_label):
    raise ValueError(
        "La longitud del corpus reducido y la longitud de las predicciones no coinciden.\n"
        f"Filas corpus_sample_df: {len(corpus_sample_df)}\n"
        f"Predicciones: {len(pred_label)}"
    )

corpus_sample_df["prob_pos"] = prob_pos_corpus_sample
corpus_sample_df["pred_label"] = pred_label

# Si existe etiqueta real, la renombramos para evitar confusión
if "labels" in corpus_sample_df.columns:
    corpus_sample_df = corpus_sample_df.rename(columns={"labels": "label_real"})

# ------------------------------------------------------------
# Guardar archivo para análisis posterior en R
# ------------------------------------------------------------

OUTPUT_CORPUS_SAMPLE_PRED = Path("corpus_sample_inference.csv")

corpus_sample_df.to_csv(
    OUTPUT_CORPUS_SAMPLE_PRED,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("\nArchivo guardado:")
print(OUTPUT_CORPUS_SAMPLE_PRED)

print("\nDimensiones del archivo final:")
print(corpus_sample_df.shape)

print("\nDistribución de predicciones:")
print(corpus_sample_df["pred_label"].value_counts(dropna=False).sort_index())

print("\nProporción predicha como tópico:")
print(corpus_sample_df["pred_label"].mean())

print("\nPrimeras filas:")
print(corpus_sample_df.head())



Archivo guardado:
corpus_sample_inference.csv

Dimensiones del archivo final:
(399, 8)

Distribución de predicciones:
pred_label
0    391
1      8
Name: count, dtype: int64

Proporción predicha como tópico:
0.020050125313283207

Primeras filas:
   row_id_corpus_sample      id  \
0                     0  144932   
1                     1  205894   
2                     2  498330   
3                     3  401111   
4                     4  647396   

                                             text_en  label_real  \
0  ' - especially when our borders are leaking an...           0   
1  ' A humane approach to smugglers during an int...           0   
2  '' Article 42: ''The State shall take special ...           0   
3  ' But the better original was a saying of Jesu...           0   
4  ' central services may not be located at their...           0   

   manifesto_id   pos  prob_pos  pred_label  
0  14820_201904   837  0.005088           0  
1  21321_201905  2051  0.005105           